# EDA - Retail Customer Analytics
## Preparation before testing dbt

**Objectif:** (Real) quick EDA,


In [ ]:
# All data bases can be found there : https://www.kaggle.com/datasets/raghavendragandhi/retail-customer-and-transaction-dataset?resource=download
# It might change depending of the quality of the data, as it a training exercice

# Rethoric :
# Cleaning data only with dbt ? I can still look for outliers, weird patterns, etc.
# How it works to clean with dbt, tests and reproductibility

In [ ]:
# Imports, if time -> plots
import pandas as pd
import os
from pathlib import Path

In [3]:
df = pd.read_csv("Data/customers.csv")
df.head()

,customer_id,full_name,age,gender,email,phone,street_address,city,state,zip_code,registration_date,preferred_channel
0,4c30e132-0704-4459-a509-9eddde934977,Mark Johnson,40.0,Male,mark.johnson@yahoo.com,989.608.3863,819 Johnson Course,Houston,Texas,29158.0,2024-04-25,NaN
1,68bec407-275f-4b5b-9a82-13d02f54626a,Robert Smith,33.0,Male,smithr@yahoo.com,(518)349-5931x0341,35116 Michael Key Suite 078,Austin,Texas,16862.0,2021-05-30,in-store
2,4466459f-76c8-433c-814e-6d59cb4131fc,Jamie Chavez,42.0,Female,jchavez@gmail.com,364.583.5030x564,419 Amanda Gardens,Detroit,Michigan,21918.0,2023-12-14,online
3,04c36a25-02f3-462c-92b0-6bf291c57706,Thomas Bradley,53.0,Male,thomas.bradley@hotmail.com,(332)887-1012x269,7242 Julie Plain Suite 969,Fort Worth,Texas,52851.0,2022-07-11,both
4,e916df3d-c3f5-40b0-8ae2-5d043be88300,Jane Ferrell,32.0,Female,jane.ferrell@hotmail.com,5484281489,845 Kelly Estate,Atlanta,Georgia,59971.0,2020-09-06,online


In [12]:
# All csv from Data
csv_files = Path("Data").glob("*.csv")

data_overview = {}

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    file_name = csv_file.stem 
    
    data_overview[file_name] = {
        'shape': df.shape,
        'columns': list(df.columns),
        'dtypes': dict(df.dtypes),
        'null_counts': dict(df.isnull().sum()),
        'sample': df.head(2).to_dict(orient='records')
    }
    
    print(f"\n{'='*60}")
    print(f"{file_name.upper()}")
    print(f"{'='*60}")
    print(f"\n Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\n Columns: {list(df.columns)}")
    print(f"\n Null values:\n{df.isnull().sum()}")
    print(f"\n Data types:\n{df.dtypes}")
    print(f"\n First 2 rows:")
    print(df.head(2).to_string())
    print("\n")


CAMPAIGNS

 Dimensions: 200 rows × 12 columns

 Columns: ['campaign_id', 'campaign_name', 'campaign_type', 'start_date', 'end_date', 'target_segment', 'budget', 'impressions', 'clicks', 'conversions', 'conversion_rate', 'roi']

 Null values:
campaign_id         0
campaign_name       6
campaign_type       6
start_date          0
end_date            0
target_segment      0
budget              4
impressions         4
clicks             10
conversions         7
conversion_rate     2
roi                 3
dtype: int64

 Data types:
campaign_id            str
campaign_name          str
campaign_type          str
start_date             str
end_date               str
target_segment         str
budget             float64
impressions        float64
clicks             float64
conversions        float64
conversion_rate    float64
roi                float64
dtype: object

 First 2 rows:
                            campaign_id                campaign_name         campaign_type  start_date    end_da

## Data Quality Issues and dbt Strategy

### Critical Data Cleaning Requirements

| Table | Rows | NULL Fields | dbt Treatment |
|-------|------|------------|----------------|
| **TRANSACTIONS** | 32,295 | product_name(678), price(622), quantity(644) | Filter or flag in staging + tests |
| **CUSTOMERS** | 5,000 | age(186), email(111), preferred_channel(114) | COALESCE + default values |
| **INTERACTIONS** | 100,000 | channel(2002), interaction_type(2022) | LEFT JOINs only, preserve all rows |
| **SUPPORT_TICKETS** | 3,000 | satisfaction_score(341), resolution_date(244) | Create flag for "unresolved" status |
| **CAMPAIGNS** | 200 | campaign_name(6), budget(4), roi(3) | Manageable, minimal nulls |
| **REVIEWS** | 1,000 | product_name(24) | Minimal, safe to filter |

---

### Key Data Architecture Observations

**Data Relationships**
```
CUSTOMERS (5k) ---> TRANSACTIONS (32k) ---> REVIEWS (1k)
      |
      +---> SUPPORT_TICKETS (3k)
      |
      +---> INTERACTIONS (100k)
      |
      +---> CAMPAIGNS (200)
```

**Key Volume Insights**
- TRANSACTIONS: 32,295 rows (central fact table)
- INTERACTIONS: 100,000 rows (high dimensional, requires aggregation)
- SUPPORT_TICKETS: 3,000 rows (manageable)
- CUSTOMERS: 5,000 rows (stable dimension)

---

### Data Cleaning Approach by Layer

**Staging Layer: Raw Data Preparation**
```sql
-- Handle NULL values explicitly
COALESCE(age, 0) AS age,
COALESCE(preferred_channel, 'unknown') AS preferred_channel,

-- Validate numeric constraints
WHERE price > 0 AND quantity > 0
```

**Key Data Integrity Checks**
1. Temporal validation: all transactions between 2020-2024 are valid
2. Constraint validation: ROI can be negative (acceptable for marketing campaigns)
3. Foreign key validation: Use LEFT JOINs to preserve all customer records
4. Satisfaction scores: 1-5 range validation
5. Resolution status: explicit flags for unresolved tickets (11% null values)

---

### dbt Implementation Strategy (3-Layer Architecture)

**Staging Layer** (data cleaning)
```
stg_customers          - COALESCE demographics, standardize channels
stg_transactions       - Validate amounts, apply discount logic
stg_support_tickets    - Create resolution flags, handle missing scores
stg_interactions       - Preserve all events, flag missing dimensions
stg_campaigns          - Minimal transformations
stg_reviews            - Label incomplete records
```

**Intermediate Layer** (business logic)
```
int_customer_metrics   - Aggregate revenue, order count, tenure per customer
int_product_performance - Category analysis, popularity scoring
int_support_quality    - Ticket volume, resolution SLA, satisfaction by category
int_interaction_summary - Session aggregation, engagement scoring
```

**Marts Layer** (analytics-ready tables)
```
fact_customer_orders         - Transaction detail with customer enrichment
fact_support_interactions    - Support tickets with customer lifetime value context
dim_customer_experience      - Customer dimension with segmentation and churn risk
dim_product_performance      - Product dimension with performance metrics
dim_marketing_campaigns      - Campaign analysis with ROI tracking
```

---

### Next Steps

1. Initialize dbt project structure (dbt init)
2. Configure database connection (profiles.yml)
3. Define data sources (schema.yml with all 6 CSV files)
4. Build staging models (stg_*.sql for each source)
5. Add automated tests (unique, not_null, expression checks)
6. Implement intermediate models (aggregations and calculations)
7. Create final mart tables (analytics-ready for Looker Studio)
8. Execute full pipeline (dbt run && dbt test)
9. Generate documentation (dbt docs generate && dbt docs serve)


In [10]:
import pandas as pd
from pathlib import Path
import json

# Load all data
csv_files = Path("Data").glob("*.csv")
data = {f.stem: pd.read_csv(f) for f in sorted(csv_files)}

print("\n" + "="*80)
print("DETAILED ANALYSIS - ANOMALIES & PATTERNS")
print("="*80)

# 1. TRANSACTIONS - Most important table
print("\nTRANSACTIONS (32,295 rows - FACT TABLE)")
print("-" * 60)
tx = data['transactions']
print(f"Price range: ${tx['price'].min():.2f} - ${tx['price'].max():.2f}")
print(f"Quantity range: {tx['quantity'].min():.0f} - {tx['quantity'].max():.0f} items")
print(f"Discount range: {tx['discount_applied'].min():.0f}% - {tx['discount_applied'].max():.0f}%")
print(f"Payment methods: {tx['payment_method'].dropna().nunique()} types")
print(f"   Breakdown: {dict(tx['payment_method'].value_counts().head(5))}")
print(f"Store locations: {tx['store_location'].nunique()} unique locations")
print(f"Products in dataset: {tx['product_name'].nunique()} unique products")
print(f"Categories: {tx['product_category'].nunique()} categories")
print(f"   Top 5: {list(tx['product_category'].value_counts().head(5).index)}")

# Check transactions with price = 0 (potential errors)
zero_price = len(tx[tx['price'] == 0])
print(f"Transactions with price=0: {zero_price} ({zero_price/len(tx)*100:.2f}%)")

print("\nREVENUE ANALYSIS")
tx_clean = tx.dropna(subset=['price', 'quantity', 'discount_applied'])
tx_clean['final_price'] = tx_clean['quantity'] * tx_clean['price'] * (1 - tx_clean['discount_applied']/100)
print(f"Total Revenue: ${tx_clean['final_price'].sum():,.2f}")
print(f"Avg Order Value: ${tx_clean['final_price'].mean():.2f}")
print(f"Median Order Value: ${tx_clean['final_price'].median():.2f}")

# 2. CUSTOMERS - Main dimension
print("\n\nCUSTOMERS (5,000 rows - DIMENSION)")
print("-" * 60)
cust = data['customers']
print(f"Age range: {cust['age'].min():.0f} - {cust['age'].max():.0f} years")
print(f"Genders: {cust['gender'].unique().tolist()}")
print(f"Registration span: {cust['registration_date'].min()} to {cust['registration_date'].max()}")
print(f"Preferred channels: {cust['preferred_channel'].dropna().unique().tolist()}")
print(f"States represented: {cust['state'].nunique()} states")
print(f"Top 5 states: {dict(cust['state'].value_counts().head(5))}")

# Customers with transactions
cust_with_tx = cust[cust['customer_id'].isin(tx['customer_id'])].shape[0]
print(f"Customers with transactions: {cust_with_tx}/5000 ({cust_with_tx/50:.1f}%)")

# 3. SUPPORT TICKETS - Customer Experience
print("\n\nSUPPORT TICKETS (3,000 rows - CUSTOMER EXPERIENCE)")
print("-" * 60)
tickets = data['support_tickets']
print(f"Unique customers with tickets: {tickets['customer_id'].nunique()}")
print(f"Issue categories: {tickets['issue_category'].nunique()}")
print(f"   Breakdown: {dict(tickets['issue_category'].value_counts().head(5))}")
print(f"Resolution status: {tickets['resolution_status'].unique().tolist()}")
print(f"   Counts: {dict(tickets['resolution_status'].value_counts())}")

tickets_with_satisfaction = tickets[tickets['customer_satisfaction_score'].notna()]
print(f"Satisfaction scores (1-5): {int(tickets_with_satisfaction['customer_satisfaction_score'].min())}-{int(tickets_with_satisfaction['customer_satisfaction_score'].max())}")
print(f"Avg satisfaction: {tickets_with_satisfaction['customer_satisfaction_score'].mean():.2f}/5")
print(f"Tickets distribution:")
print(f"   5 stars: {len(tickets_with_satisfaction[tickets_with_satisfaction['customer_satisfaction_score']==5])}")
print(f"   4 stars: {len(tickets_with_satisfaction[tickets_with_satisfaction['customer_satisfaction_score']==4])}")
print(f"   3 stars: {len(tickets_with_satisfaction[tickets_with_satisfaction['customer_satisfaction_score']==3])}")
print(f"   Less than 3 stars: {len(tickets_with_satisfaction[tickets_with_satisfaction['customer_satisfaction_score']<3])}")

# Resolution time
tickets_resolved = tickets[tickets['resolution_time_hours'].notna()]
print(f"Avg resolution time: {tickets_resolved['resolution_time_hours'].mean():.1f} hours")
print(f"Max resolution time: {tickets_resolved['resolution_time_hours'].max():.1f} hours")

# 4. INTERACTIONS - Behavior tracking (huge!)
print("\n\nINTERACTIONS (100,000 rows - HIGH VOLUME TABLE)")
print("-" * 60)
interactions = data['interactions']
print(f"Unique customers: {interactions['customer_id'].nunique()}")
print(f"Channels: {interactions['channel'].dropna().unique().tolist()}")
print(f"Interaction types: {interactions['interaction_type'].dropna().unique().tolist()}")
print(f"Avg session duration: {interactions['duration'].mean():.1f} seconds")
print(f"Max session duration: {interactions['duration'].max():.1f} seconds")

# Sessions analysis
sessions = interactions.groupby('session_id').size()
print(f"Unique sessions: {interactions['session_id'].nunique()}")
print(f"Avg events per session: {sessions.mean():.1f}")

# 5. MARKETING CAMPAIGNS
print("\n\nCAMPAIGNS (200 rows)")
print("-" * 60)
campaigns = data['campaigns']
print(f"Campaign types: {campaigns['campaign_type'].unique().tolist()}")
print(f"Total budget: ${campaigns['budget'].sum():,.0f}")
print(f"Avg ROI: {campaigns['roi'].mean():.1f}%")
print(f"Best ROI: {campaigns['roi'].max():.1f}%")
print(f"Worst ROI: {campaigns['roi'].min():.1f}%")
print(f"Campaigns with negative ROI: {len(campaigns[campaigns['roi']<0])}")

# 6. REVIEWS
print("\n\nREVIEWS (1,000 rows)")
print("-" * 60)
reviews = data['customer_reviews_complete']
print(f"Rating distribution:")
print(f"   {dict(reviews['rating'].value_counts().sort_index(ascending=False))}")
print(f"Avg rating: {reviews['rating'].mean():.2f}/5")
print(f"Products reviewed: {reviews['product_name'].nunique()}")
print(f"Review length (avg chars): {reviews['review_text'].str.len().mean():.0f}")

print("\n" + "="*80)
print("READY FOR DBT MODELING!")
print("="*80)



DETAILED ANALYSIS - ANOMALIES & PATTERNS

TRANSACTIONS (32,295 rows - FACT TABLE)
------------------------------------------------------------
Price range: $17.45 - $21371.65
Quantity range: 1 - 50 items
Discount range: 0% - 30%
Payment methods: 7 types
   Breakdown: {'Credit Card': np.int64(10907), 'Debit Card': np.int64(7968), 'PayPal': np.int64(4798), 'Apple Pay': np.int64(3160), 'Google Pay': np.int64(1633)}
Store locations: 11 unique locations
Products in dataset: 75 unique products
Categories: 15 categories
   Top 5: ['Smartphones', 'Smart Home Devices', 'Furniture', 'Home Decor', 'Kitchen Appliances']
Transactions with price=0: 0 (0.00%)

REVENUE ANALYSIS
Total Revenue: $25,793,453.22
Avg Order Value: $846.88
Median Order Value: $466.17


CUSTOMERS (5,000 rows - DIMENSION)
------------------------------------------------------------
Age range: 18 - 80 years
Genders: ['Male', 'Female', 'Non-binary', 'Prefer not to say', nan]
Registration span: 2020-02-28 to 2025-02-26
Preferred 